<a href="https://colab.research.google.com/github/abdulstarzeki001-sketch/linux-terminal/blob/main/colab/STAR_AI_ULTRA_GPU_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STAR AI ULTRA GPU Colab — Qwen3.6 A100 80GB

**نموذج برمجة وذكاء اصطناعي محلي على GPU السحابي.** هذه نسخة مستقلة لا تعدّل دفتر STAR AI ULTRA FAST CODER أو ملفك الشخصي على Google Drive.

- **النموذج الأساسي عند توفر الموارد:** `qwen3.6:35b` — تنزيل قرابة 23GB حسب Ollama.
- بديل أصغر: `qwen3.6:27b`؛ وعند انخفاض الموارد: `qwen3.5:9b` أو `qwen3.5:4b`.
- اختر `Runtime → Change runtime type → A100 GPU → High-RAM` إن كانت متاحة. سرعة التشغيل الفعلية تعتمد على VRAM والقرص والسياق.
- لا يحتاج مفتاح API من Kimi أو خدمات خارجية. **لا يقوم بالتحكم في لابتوبك أو Railway**؛ يعمل في حاوية Colab المؤقتة فقط.
- لا تنفّذ أوامر يقترحها النموذج مباشرة؛ التشخيص محصور بأوامر قراءة ثابتة.
- خدمة Ollama تستمع فقط على `127.0.0.1:11434` ولا يتم نشر API عام.
- تشغيل الخلايا قد يستهلك وحدات Colab المدفوعة وينزّل عشرات الجيجابايت. لا تخزن مفاتيح أو بيانات خاصة داخل الدفتر.
- نجاح اختبارات GitHub يثبت صحة الصياغة فقط، وليس أن النموذج نفّذ استدلالاً على A100.

معلومات النموذج: https://ollama.com/library/qwen3.6

## 1) تحقق فعلي من GPU والذاكرة والمساحة
يُرفض التشغيل دون NVIDIA GPU، وتُقرأ الموارد من الجلسة الحالية بدلاً من افتراض سعة ثابتة.

In [1]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

def run_cmd(args, timeout=30):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout, check=False)

gpu_check = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"])
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("No NVIDIA GPU detected. Select Runtime → Change runtime type → GPU.")

gpu_name, total_mib, free_mib = [
    v.strip() for v in gpu_check.stdout.splitlines()[0].rsplit(",", 2)
]
gpu_free_gib = int(free_mib) / 1024
disk_free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print("GPU:", gpu_name)
print("GPU total GiB:", round(int(total_mib) / 1024, 1))
print("GPU free GiB:", round(gpu_free_gib, 1))
print("Disk free GiB:", round(disk_free_gib, 1))
print("Python:", sys.version.split()[0])
print("Runtime location: Colab cloud; your laptop is not executing this model.")

GPU: NVIDIA L4
GPU total GiB: 22.5
GPU free GiB: 22.0
Disk free GiB: 235.6
Python: 3.13.16
Runtime location: Colab cloud; your laptop is not executing this model.


## 2) اختيار نموذج Qwen3.6 تلقائياً

يختار الدفتر `qwen3.6:35b` عندما تسمح ذاكرة GPU والقرص. إن كانت الموارد أقل، يختار بدائل أقل حجماً دون تغيير أي من دفاتر المستودع الأخرى. المخصصات أدناه محافظة لكنها ليست ضماناً للتوافق؛ نتيجة `ollama ps` تحسم ما إذا كان التنفيذ على GPU.

In [2]:
MODEL_OVERRIDE = None  # Optional example: "qwen3.6:27b"
# (minimum free GPU GiB, minimum free disk GiB) before downloads
MODEL_CHOICES = [
    ("qwen3.6:35b", 34, 33),
    ("qwen3.6:27b", 27, 27),
    ("qwen3.5:9b", 12, 12),
    ("qwen3.5:4b", 7, 8),
]
eligible = [name for name, need_gpu, need_disk in MODEL_CHOICES
            if gpu_free_gib >= need_gpu and disk_free_gib >= need_disk]
if not eligible:
    raise RuntimeError("Insufficient free GPU memory or disk for the configured models")
if MODEL_OVERRIDE is not None and MODEL_OVERRIDE not in eligible:
    raise ValueError(f"Override is not eligible for measured resources: {eligible}")
MODEL = MODEL_OVERRIDE or eligible[0]
CONTEXT = 8192
print("Selected Qwen model:", MODEL)
print("Eligible models:", eligible)
print("Model download has NOT started yet.")

Selected Qwen model: qwen3.5:9b
Eligible models: ['qwen3.5:9b', 'qwen3.5:4b']
Model download has NOT started yet.


## 3) تثبيت Ollama من المصدر الرسمي
يحمّل السكربت الرسمي من `ollama.com`. افحص سياسة الثقة في مصادر التثبيت قبل تشغيله في أي بيئة مؤسسية.

In [ ]:
import shutil
import subprocess
import os
import urllib.request
import time

# Force clean any previous installations to ensure clean upgrade
for path in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/tmp/ollama", "/tmp/ollama-linux-amd64.tgz", "/tmp/bin"]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

print("Installing latest Ollama version (v0.5.11) via Python robust downloader...")
url = "https://github.com/ollama/ollama/releases/download/v0.5.11/ollama-linux-amd64.tgz"
dest = "/tmp/ollama-linux-amd64.tgz"

# Download with up to 3 retries
for attempt in range(3):
    try:
        print(f"Download attempt {attempt+1}...")
        urllib.request.urlretrieve(url, dest)
        print("Download completed successfully.")
        break
    except Exception as e:
        if attempt == 2:
            raise e
        print(f"Error: {e}. Retrying in 5 seconds...")
        time.sleep(5)

# Extract the archive
subprocess.run(["tar", "-C", "/tmp", "-xzf", dest], check=True)
# Move the extracted binary to local bin
subprocess.run(["mv", "/tmp/bin/ollama", "/usr/local/bin/ollama"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/ollama"], check=True)

if not shutil.which("ollama"):
    raise RuntimeError("Ollama installation failed")

print("Ollama version:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

Installing latest Ollama version (v0.5.11) via Python robust downloader...
Download attempt 1...


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import subprocess
import shutil
import os

ollama_path = shutil.which("ollama")
if not ollama_path:
    if os.path.exists("/usr/local/bin/ollama"):
        ollama_path = "/usr/local/bin/ollama"
    elif os.path.exists("/usr/bin/ollama"):
        ollama_path = "/usr/bin/ollama"

if ollama_path:
    try:
        result = subprocess.run([ollama_path, "--version"], capture_output=True, text=True, check=True)
        print("✅ Installed Ollama Version:")
        print(result.stdout.strip())
    except Exception as e:
        print("❌ Failed to execute Ollama.")
        print(e)
else:
    print("❌ Ollama executable not found! The installation might have failed or hasn't finished yet.")

❌ Ollama executable not found! The installation might have failed or hasn't finished yet.


In [16]:
import subprocess
import os

ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"

print("Starting pull for qwen3.5:9b...")
result = subprocess.run(["ollama", "pull", "qwen3.5:9b"], env=ollama_env, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

Starting pull for qwen3.5:9b...

Errors: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠋ pulling manifest 
Error: pull model manifest: 412: 
The model you are attempting to pull requires a newer version of Ollama.

Please download the latest version at:

	https://ollama.com/download




## 4) تشغيل الخادم داخل جلسة Colab فقط
لا تستخدم نفقاً عاماً أو رابطاً بدون مصادقة. تم ضبط مضيف Ollama على localhost حصراً.

In [5]:
import requests
import os
import subprocess
import time
import shutil
from pathlib import Path

# Force kill any existing old Ollama processes to use the newly installed version
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_URL = "http://" + OLLAMA_HOST
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_CONTEXT_LENGTH": str(globals().get("CONTEXT", 8192)),
    "OLLAMA_KEEP_ALIVE": "5m",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
})

ollama_path = shutil.which("ollama")
if not ollama_path:
    if os.path.exists("/usr/local/bin/ollama"):
        ollama_path = "/usr/local/bin/ollama"
    elif os.path.exists("/usr/bin/ollama"):
        ollama_path = "/usr/bin/ollama"
    else:
        raise FileNotFoundError("❌ Ollama executable not found! Please run the installation cell (6zXoPm82LGjW) first.")

def ollama_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_ready():
    print("Starting local Ollama server in background...")
    log_file = open("/tmp/star-ai-ultra-ollama.log", "a")
    ollama_process = subprocess.Popen(
        [ollama_path, "serve"], env=ollama_env,
        stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True
    )
    for i in range(30):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(1)

if not ollama_ready():
    tail = Path("/tmp/star-ai-ultra-ollama.log").read_text(errors="replace")[-2500:]
    raise RuntimeError("Ollama failed to start. Log tail:\n" + tail)

print("Local Ollama ready:", OLLAMA_URL)

FileNotFoundError: ❌ Ollama executable not found! Please run the installation cell (6zXoPm82LGjW) first.

In [8]:
import requests

# حدد اسم النموذج الذي تريد تشغيله
target_model = "qwen3.5:9b"
prompt = "اكتب فقرة قصيرة تشرح فيها ما هو الذكاء الاصطناعي."

print(f"Running model: {target_model}...")

try:
    # إرسال الطلب إلى خادم Ollama المحلي
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": target_model,
            "prompt": prompt,
            "stream": False # اجعلها True إذا كنت تريد تدفق النص (Streaming)
        }
    )
    response.raise_for_status()

    result = response.json()
    print("\nالرد (Response):")
    print(result.get("response", ""))

except requests.exceptions.RequestException as e:
    print(f"❌ حدث خطأ أثناء الاتصال بالخادم. تأكد من أن الخادم يعمل وأن النموذج تم تنزيله: {e}")

Running model: qwen3.5:9b...
❌ حدث خطأ أثناء الاتصال بالخادم. تأكد من أن الخادم يعمل وأن النموذج تم تنزيله: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e78acaeacf0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [2]:
import requests
import subprocess

print("CLI Version:")
subprocess.run(["/usr/local/bin/ollama", "--version"])

try:
    res = requests.get("http://127.0.0.1:11434/api/version")
    print("\nServer Version:", res.json())
except Exception as e:
    print("\nCould not get server version:", e)

CLI Version:


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [9]:
import requests
import json

try:
    # Querying the local Ollama health check / API tags endpoint
    response = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
    if response.status_code == 200:
        print("✅ Ollama Server is UP and RUNNING!")
        print("Response JSON:")
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"⚠️ Ollama Server responded with unexpected status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Ollama Server is DOWN! (Could not connect to http://127.0.0.1:11434)")
except Exception as e:
    print(f"❌ An error occurred during health check: {e}")

❌ Ollama Server is DOWN! (Could not connect to http://127.0.0.1:11434)


In [12]:
import os
from pathlib import Path

log_path = Path("/tmp/star-ai-ultra-ollama.log")

if log_path.exists():
    print("--- Ollama Server Log Tail (Last 40 lines) ---")
    lines = log_path.read_text(errors="replace").splitlines()
    for line in lines[-40:]:
        print(line)
else:
    print("❌ Log file does not exist. The server startup cell (JTpPB2JGLGjY) hasn't been executed yet or was stopped.")

❌ Log file does not exist. The server startup cell (JTpPB2JGLGjY) hasn't been executed yet or was stopped.


In [13]:
import subprocess
result = subprocess.run(["ps", "aux"], capture_output=True, text=True)
ollama_processes = [line for line in result.stdout.splitlines() if "ollama" in line.lower()]
if ollama_processes:
    print("--- Active Ollama Processes Found ---")
    for proc in ollama_processes:
        print(proc)
else:
    print("❌ No running Ollama processes found.")

❌ No running Ollama processes found.


In [6]:
import subprocess
import shutil
import requests
import time
import os

current_model = globals().get("MODEL", "qwen3.5:9b")

ollama_path = shutil.which("ollama")
if not ollama_path:
    if os.path.exists("/usr/local/bin/ollama"):
        ollama_path = "/usr/local/bin/ollama"
    elif os.path.exists("/usr/bin/ollama"):
        ollama_path = "/usr/bin/ollama"
    else:
        raise FileNotFoundError("❌ Ollama executable not found! Please run the installation cell (6zXoPm82LGjW) first.")

# Configure environments
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
})

print("Checking Ollama server status before pulling...")
server_ready = False
for attempt in range(15):
    try:
        res = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if res.ok:
            server_ready = True
            break
    except requests.RequestException:
        pass
    print(f"Waiting for Ollama server to boot up... (Attempt {attempt + 1}/15)")
    time.sleep(2)

if not server_ready:
    raise RuntimeError("❌ Ollama server is not running! Please run the server start cell (JTpPB2JGLGjY) first.")

print(f"✅ Server is active. Starting download for model: {current_model}...")
print(f"Available Disk Space: {round(shutil.disk_usage('/content').free / (1024**3), 2)} GiB")

try:
    result = subprocess.run([ollama_path, "pull", current_model], env=ollama_env, check=True, text=True, capture_output=True)
    print(result.stdout)
    print(f"Successfully pulled model: {current_model}!")
except subprocess.CalledProcessError as e:
    print(f"❌ Failed to pull model {current_model}.")
    print(e.stderr)

FileNotFoundError: ❌ Ollama executable not found! Please run the installation cell (6zXoPm82LGjW) first.

## 5) تنزيل الأوزان (عند التشغيل)

الحجم المعلن لنموذج `qwen3.6:35b` قرابة **23GB**. قد تحتاج مساحة إضافية للتنزيل والتشغيل. تتكرر عملية التنزيل بعد انتهاء جلسة Colab إذا لم تحفظ الأوزان في مخزن دائم. لا تعرض سجل التنزيل كمؤشر على نجاح الاستدلال.

In [ ]:
print("Downloading model:", MODEL)
print("Available storage GiB before download:", round(shutil.disk_usage("/content").free / 2**30, 1))
subprocess.run(["ollama", "pull", MODEL], check=True, env=ollama_env)
print("Downloaded successfully:", MODEL)

## 6) اختبار محادثة حقيقي + إثبات هل التنفيذ على GPU
رسالة الرد وحدها لا تثبت استخدام GPU؛ يتم عرض `ollama ps` و`nvidia-smi` بعد تنفيذ الاستدلال.

In [ ]:
SYSTEM = (
    "You are STAR AI ULTRA Qwen3.6, an Arabic-first software engineer and code reviewer "
    "for Python, TypeScript, Node.js and authorized defensive security tasks. Be exact; never claim to have run "
    "commands or accessed external systems unless a provided tool result proves it. "
    "For any impactful system change, require user approval."
)

def ask_model(messages, max_tokens=512):
    r = requests.post(
        OLLAMA_URL + "/api/chat",
        json={
            "model": MODEL, "messages": messages, "stream": False,
            "think": False,
            "options": {"num_ctx": CONTEXT, "num_predict": max_tokens,
                        "temperature": 0.35}, "keep_alive": "5m"
        },
        timeout=600
    )
    r.raise_for_status()
    payload = r.json()
    answer = (payload.get("message") or {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("Model returned no text; inspect /tmp/star-ai-ultra-ollama.log")
    return answer, payload

reply, stats = ask_model([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "اكتب سطرين بالعربية عن تحليل سجلات Linux الدفاعي."}
], max_tokens=180)
print("MODEL RESPONSE:\n", reply)
print("\nMODEL:", stats.get("model"))
print("Generated tokens:", stats.get("eval_count"))
if stats.get("eval_duration") and stats.get("eval_count"):
    print("Generation tokens/sec:", round(stats["eval_count"] / (stats["eval_duration"] / 1e9), 2))
print("\nGPU/CPU processor report (check PROCESSOR column):")
processor = subprocess.run(["ollama", "ps"], env=ollama_env,
                           capture_output=True, text=True, check=True).stdout
print(processor)
if "GPU" not in processor.upper():
    raise RuntimeError("Model generated a response, but GPU execution was NOT verified. Inspect ollama ps and model allocation.")
print("\nCurrent GPU usage:")
print(run_cmd(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
               "--format=csv,noheader"]).stdout.strip())

## 7) محادثة تفاعلية مع STAR AI
اكتب `chat("سؤالك")` لتتحدث مع النموذج. حفظ الذاكرة هنا مؤقت داخل الجلسة وليس في GitHub أو Drive.

In [ ]:
history = [{"role": "system", "content": SYSTEM}]

def chat(question, max_tokens=1000):
    global history
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must not be empty")
    answer, _ = ask_model(
        history + [{"role": "user", "content": question.strip()}],
        max_tokens=max_tokens
    )
    history.extend([
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": answer}
    ])
    history = [history[0]] + history[-10:]
    print("STAR AI ULTRA:\n" + answer)
    return answer

chat("صمم خدمة Python FastAPI لإدارة مهام STAR AI مع مصادقة JWT، تحقق صلاحيات، وسجل تدقيق، مع اختبارين pytest.", max_tokens=700)

## 7A) مراجعة ملفات STAR AI من مساحة العمل (قراءة فقط)
استنسخ مشروعك العام إلى `/content/star-ai-workspace` إذا احتجت تحليل الملفات. لا تضع الأسرار في المشروع العام، ولا تشغّل تلقائياً أوامر من المساعد. هذه الخلية لا تغيّر ملفات المشروع، وتحمي من قراءة المسارات خارج مساحة العمل.

In [ ]:
from pathlib import Path
WORKSPACE = Path("/content/star-ai-workspace").resolve()
PUBLIC_REPO = "https://github.com/abdulstarzeki001-sketch/linux-terminal.git"
CLONE_PROJECT = False  # Change to True only to clone your authorized public repo
if CLONE_PROJECT:
    if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
        print("Workspace already exists; no changes made")
    else:
        subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO, str(WORKSPACE)],
                       check=True, timeout=120)
        print("Public repo cloned into", WORKSPACE)
else:
    print("Repository clone disabled by default.")

def review_local_file(relative_file):
    import re
    path = (WORKSPACE / relative_file).resolve()
    if not WORKSPACE.exists() or not path.is_relative_to(WORKSPACE) or not path.is_file():
        raise ValueError("File must exist inside /content/star-ai-workspace")
    if path.is_symlink() or any(
        part.startswith(".env") or re.search(r"(secret|credential|token|private|id_rsa|pem|key)", part, re.I)
        for part in path.parts[len(WORKSPACE.parts):]
    ):
        raise ValueError("Potential secret or sensitive path is not allowed")
    if path.stat().st_size > 40000:
        raise ValueError("File too large; select a smaller file for review")
    data = path.read_text(encoding="utf-8", errors="replace")
    if len(data) > 25000:
        data = data[:25000] + "\\n[truncated]"
    prompt = ("Review the following code for correctness, maintainability and security. "
              "Return prioritized fixes and suggested diffs; do not claim to edit files. "
              "FILE: " + relative_file + "\\n" + data)
    answer, _ = ask_model([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt}
    ], max_tokens=1400)
    print(answer)
    return answer

# Example after manually enabling clone:
# review_local_file("server/app.py")

## 8) تشخيص Linux الحقيقي — أوامر محددة وآمنة
هذه الأوامر تُنفّذ فعلياً على **خادم Colab**، وليس لابتوبك أو Railway. لا يُشغّل أي نص يأتي من النموذج كأمر shell.

In [ ]:
SAFE_DIAGNOSTICS = {
    "gpu": ["nvidia-smi"],
    "ram": ["free", "-h"],
    "disk": ["df", "-h", "/content"],
    "python": ["python", "--version"],
    "ollama": ["ollama", "ps"],
}

def diagnostic(name):
    if name not in SAFE_DIAGNOSTICS:
        raise ValueError("Allowed diagnostics: " + ", ".join(SAFE_DIAGNOSTICS))
    result = subprocess.run(SAFE_DIAGNOSTICS[name], shell=False,
                            capture_output=True, text=True, timeout=30,
                            env=ollama_env)
    print(result.stdout or result.stderr)
    return result.returncode

diagnostic("gpu")

## 9) حفظ محادثة غير حساسة اختيارياً في Google Drive
بشكل افتراضي **لا يتم تركيب Drive ولا حفظ محادثات**. إذا أردت الحفظ، غيّر `SAVE_TO_DRIVE=True`؛ سيطلب Colab تفويض حسابك. راجع المحتوى قبل الحفظ، ولا تحفظ بيانات عملاء أو مفاتيح API.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/STAR_AI_ULTRA")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "chat_history.json"
    output_file.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", output_file)
else:
    print("Drive persistence disabled. No chat files written.")

## 10) إيقاف التشغيل والأمان

- عند الانتهاء: Runtime → Disconnect and delete runtime لإنهاء استهلاك GPU.
- Colab مؤقت وليس خدمة ذكاء اصطناعي دائمة. لا تفتح Ollama للعامة أو تربط المنفذ مباشرة بخدمات Railway/Cloudflare.
- بعد رسالة الاختبار، افحص `ollama ps`. إذا لم يظهر GPU في عمود PROCESSOR، فلا تعتبر التسريع ناجحاً.
- إذا نفدت ذاكرة GPU أو المساحة، غيّر `MODEL_OVERRIDE` إلى `qwen3.6:27b` أو `qwen3.5:9b` ضمن النماذج المؤهلة، ثم أعد التحميل والاختبار.
- فحص ملفات GitHub محلياً للقراءة فقط؛ لا تمنح النموذج صلاحية تشغيل أوامر نظام غير مُراجعة.
- لا تضع مفاتيح API أو بيانات خاصة في Notebook عام.

### مصادر
- https://ollama.com/library/qwen3.6
- https://ollama.com/library/qwen3.5
- https://docs.ollama.com/api
- https://research.google.com/colaboratory/faq.html